# X5_Q1 — Tabla maestra + variables de estado del activo

Primer notebook de **X5_alt_v2** (evolución paralela de X5_alt — ver
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10,
documento rector [`docs/plans/x5_contexto_conversacion.md`](../docs/plans/x5_contexto_conversacion.md)).
No reemplaza `X5_P1.ipynb`/`X5_P2.ipynb` (v1, congelados en `scripts/otros_strings/`).

Este notebook construye, para cada activo de `valores`, la misma tabla
continua en el tiempo (una fila por vela H1) que ya resolvía `X5_P1.ipynb` —
precio + X3 (técnicos) + X2 (fundamental) — y la deja como punto de partida
para las variables de estado del activo (drawdown, volatilidad, ruido,
tendencia/régimen, liquidez) que se agregan en los ítems siguientes de
`X5_alt_v2` (TODOs en `docs/tracking/todos.md`, sección X5_alt_v2).

**Input único que se puede cambiar — lista, no string** (a diferencia del
`valor` singular de v1): soporta varios activos sin reescribir esta celda.

In [1]:
valores = ['BTCUSD']  # lista de activos; default solo BTCUSD (ver plan X5_alternativo.md §10.2)

## 0. Preparación

Reutilizamos `_calcular_todos_indicadores` de `X3_technical_features.py`
(igual que `X5_P1.ipynb`) en vez de reimplementar los indicadores técnicos.

In [2]:
import json  # para leer x2_history.json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402
from X3_technical_features import _calcular_todos_indicadores  # noqa: E402  (reusa X3 en vez de reimplementar indicadores)

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

print(f"Activos: {valores}")

Activos: ['BTCUSD']


## 1. Tabla base por activo (precio + X3 + X2)

Misma lógica de ensamblado que `X5_P1.ipynb` (precio OHLCV + indicadores
técnicos de X3 + fundamental de X2 con forward-fill), encapsulada en una
función para poder correrla sobre cada activo de `valores`.

In [3]:
def _cargar_tabla_base(valor):
    # Precio: el CSV no viene ordenado por fecha y puede traer velas duplicadas (ver X5_P1 §1)
    path_precio = cfg.CARPETA_DATA / f'{valor}.csv'
    df_precio = pd.read_csv(path_precio)
    df_precio['DateTime'] = pd.to_datetime(df_precio['DateTime'])
    df_precio = (df_precio.sort_values('DateTime')
                           .drop_duplicates(subset=['DateTime'])
                           .reset_index(drop=True))

    # X3 sobre el histórico completo (no incremental) para que indicadores con
    # warm-up largo (EMA, RSI, SMA 200) sean correctos desde la primera vela útil.
    # conjunto_N vacío = sin distancia a soportes (fuera de v0, igual que X5_P1 §2).
    df_x3 = _calcular_todos_indicadores(df_precio, set())
    df_x3 = df_x3.drop(columns=['dist_nearest_support', 'dist_floor_support', 'density_2pct'])
    df_x3 = df_x3.rename(columns={c: f'x3_{c}' for c in df_x3.columns if c != 'datetime'})

    # X2: x2_history.json trae todos los activos mezclados — filtramos por {valor}
    path_x2 = cfg.CARPETA_FUNDAMENTALS / 'x2_history.json'
    with open(path_x2) as f:
        x2_hist = json.load(f)
    registros = [e for e in x2_hist if e.get('activo') == valor]
    df_x2 = pd.DataFrame([
        {
            'DateTime': pd.to_datetime(e.get('periodo_inicio') or e['date']),
            'x2_score': e.get('score'),
            'x2_score_cross': e.get('score_cross'),
            'x2_score_tendencia': e.get('score_tendencia'),
        }
        for e in registros
    ]).sort_values('DateTime').reset_index(drop=True)  # merge_asof exige orden ascendente

    # Precio y X3 comparten índice/orden → join exacto por DateTime
    tabla = df_precio.merge(df_x3, left_on='DateTime', right_on='datetime').drop(columns=['datetime'])
    # merge_asof(direction='backward') = forward-fill del último score de X2 conocido
    tabla = pd.merge_asof(tabla, df_x2, on='DateTime', direction='backward')
    return tabla


tablas = {valor: _cargar_tabla_base(valor) for valor in valores}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas — {tabla['DateTime'].min()} → {tabla['DateTime'].max()}")

BTCUSD: 40283 filas x 37 columnas — 2021-10-27 16:00:00 → 2026-06-02 21:00:00


## 2. Técnicos filtrados y transformados (Q1-3)

`X3_technical_features.py` ya calcula niveles crudos de medias móviles
(`x3_sma_{20,50,200}`, `x3_ema_{12,26}`) — triviales como variable de estado
porque viven en la escala del propio precio (correlación ~1 con `Close` por
construcción, no aportan señal nueva). Se descartan esos niveles crudos y se
reemplazan por dos transformaciones normalizadas por ATR (independientes de
la escala de precio y del régimen de volatilidad, a diferencia de
`x3_sma_dist_{w}`/`x3_ema_dist_{w}` que ya existen normalizados por `Close`):

- `distancia_sma_{w}` / `distancia_ema_{w}` = `(Close - media_{w}) / ATR_14`
- `pendiente_sma_{w}` / `pendiente_ema_{w}` = cambio promedio por período de
  la media móvil en los últimos `w` períodos, normalizado por `ATR_14`

In [4]:
def _filtrar_y_transformar_tecnicos(df):
    atr = df['x3_atr_14']
    for w in cfg.X3_VENTANAS['sma']:
        col = f'x3_sma_{w}'
        df[f'distancia_sma_{w}'] = (df['Close'] - df[col]) / atr
        df[f'pendiente_sma_{w}'] = (df[col].diff(w) / w) / atr
    for w in cfg.X3_VENTANAS['ema']:
        col = f'x3_ema_{w}'
        df[f'distancia_ema_{w}'] = (df['Close'] - df[col]) / atr
        df[f'pendiente_ema_{w}'] = (df[col].diff(w) / w) / atr

    # niveles crudos descartados (trivial vs. Close); se conservan x3_sma_dist_*/x3_ema_dist_* (ya normalizados por Close)
    cols_crudas = [f'x3_sma_{w}' for w in cfg.X3_VENTANAS['sma']] + [f'x3_ema_{w}' for w in cfg.X3_VENTANAS['ema']]
    return df.drop(columns=cols_crudas)


tablas = {valor: _filtrar_y_transformar_tecnicos(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    nuevas = [c for c in tabla.columns if c.startswith(('distancia_', 'pendiente_'))]
    print(f"{valor}: {tabla.shape[1]} columnas — agregadas {nuevas}")

BTCUSD: 42 columnas — agregadas ['distancia_sma_20', 'pendiente_sma_20', 'distancia_sma_50', 'pendiente_sma_50', 'distancia_sma_200', 'pendiente_sma_200', 'distancia_ema_12', 'pendiente_ema_12', 'distancia_ema_26', 'pendiente_ema_26']


## 3. Variables de drawdown del activo (Q1-4)

Drawdown **del activo** (nunca de cuenta/posiciones — eso es responsabilidad
de X1/X4). Dos vistas complementarias:

- **Histórico**: `drawdown_historico` contra el máximo histórico acumulado
  (`cummax`), más `duracion_drawdown` (períodos desde el último máximo),
  `velocidad_caida` (profundidad / duración: caída promedio por período desde
  el máximo) y `recuperacion_desde_minimo` (cuánto se recuperó el precio desde
  el mínimo del episodio de drawdown actual — un episodio nuevo empieza en
  cada máximo histórico).
- **Por ventana**: `drawdown_7d/30d/90d/365d` contra el máximo rodante de esa
  ventana — a diferencia del histórico, "resetea" la referencia si el precio
  alcanzó un máximo reciente aunque no sea el máximo histórico absoluto.

In [5]:
def _variables_drawdown(df):
    close = df['Close']
    cummax = close.cummax()
    df['drawdown_historico'] = (close - cummax) / cummax

    nuevo_maximo = close >= cummax
    episodio = nuevo_maximo.cumsum()
    df['duracion_drawdown'] = close.groupby(episodio).cumcount()
    min_episodio = close.groupby(episodio).cummin()
    df['recuperacion_desde_minimo'] = (close - min_episodio) / min_episodio
    df['velocidad_caida'] = df['drawdown_historico'] / df['duracion_drawdown'].replace(0, np.nan)

    for dias, horas in [(7, 168), (30, 720), (90, 2160), (365, 8760)]:
        rolling_max = close.rolling(horas, min_periods=1).max()
        df[f'drawdown_{dias}d'] = (close - rolling_max) / rolling_max

    return df


tablas = {valor: _variables_drawdown(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[1]} columnas — drawdown_historico en [{tabla['drawdown_historico'].min():.3f}, {tabla['drawdown_historico'].max():.3f}], "
          f"duracion_drawdown max {tabla['duracion_drawdown'].max()} períodos")

BTCUSD: 50 columnas — drawdown_historico en [-0.772, 0.000], duracion_drawdown max 20296 períodos


## 4. Variables de volatilidad (Q1-5)

ATR absoluto/relativo y volatilidad realizada corta/larga ya los calcula X3
(`x3_atr_14`, `x3_atr_pct`, `x3_vol_24h`, `x3_vol_7d`) — se reutilizan tal
cual, sin reimplementar. Se agregan 3 variables nuevas, todas sobre retornos
logarítmicos horarios:

- `cambio_volatilidad` = `vol_24h / vol_7d` — >1 indica un salto de
  volatilidad reciente sobre la línea base de 7 días.
- `vol_alcista_30d` / `vol_bajista_30d` = desviación estándar anualizada de
  solo los retornos positivos / negativos en una ventana de 30 días — separa
  el riesgo al alza del riesgo a la baja (una serie casi simétrica en retorno
  medio puede no serlo en volatilidad direccional).
- `frecuencia_extremos_30d` = fracción de horas, en los últimos 30 días, con
  `|retorno log| > 2 × desviación estándar expandida hasta ese punto` — el
  umbral usa `expanding()` (no rolling) para no mirar información futura al
  definirlo; proxy de cuán seguido el activo tiene movimientos atípicos.

In [6]:
def _variables_volatilidad(df):
    close = df['Close']
    log_ret = np.log(close / close.shift(1))

    df['cambio_volatilidad'] = df['x3_vol_24h'] / df['x3_vol_7d']

    ret_up = log_ret.where(log_ret > 0)
    ret_down = log_ret.where(log_ret < 0)
    df['vol_alcista_30d'] = ret_up.rolling(720, min_periods=50).std(ddof=1) * np.sqrt(8760)
    df['vol_bajista_30d'] = ret_down.rolling(720, min_periods=50).std(ddof=1) * np.sqrt(8760)

    umbral = 2 * log_ret.expanding(min_periods=100).std(ddof=1)
    extremo = log_ret.abs() > umbral
    df['frecuencia_extremos_30d'] = extremo.rolling(720, min_periods=50).mean()

    return df


tablas = {valor: _variables_volatilidad(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[1]} columnas — cambio_volatilidad media {tabla['cambio_volatilidad'].mean():.3f}, "
          f"frecuencia_extremos_30d media {tabla['frecuencia_extremos_30d'].mean():.3f}")

BTCUSD: 54 columnas — cambio_volatilidad media 0.938, frecuencia_extremos_30d media 0.039


## 5. Variables de liquidez (Q1-8)

`Spread` y `Tick_Volume` vienen directo del CSV de MT5 (`Real_Volume` siempre
en 0 — broker no lo reporta, ver `CLAUDE.md`). Slippage no se agrega: no
existe en los datos de `Data/` — es una métrica de ejecución (brecha OE→OA)
que vive en los logs de X1/X4, no en la tabla de estado del activo.

- `spread_relativo` = `Spread / media_móvil_30d(Spread)` — `Spread` viene en
  puntos del bróker MT5 (requiere `symbol_info(valor).point` para pasarlo a
  precio, dato no disponible en Mac sin MT5 ni persistido en los CSVs), así
  que no se puede dividir por `Close` sin asumir un factor de conversión no
  verificable. Normalizar contra su propio promedio móvil evita ese supuesto:
  >1 indica spread más ancho que lo habitual reciente para ese activo.
- `tick_volume_z` = z-score de `Tick_Volume` contra media/desviación móviles
  de 30 días — normaliza el volumen frente a la tendencia de crecimiento de
  actividad del activo en el tiempo, no solo frente al nivel reciente.
- `volumen_relativo` = `Tick_Volume / media_móvil_30d(Tick_Volume)` — volumen
  relativo (RVOL): >1 indica actividad por sobre su propio promedio reciente,
  útil para detectar picos puntuales que el z-score (más sensible a la
  dispersión) puede diluir.

In [7]:
def _variables_liquidez(df):
    spread = df['Spread']
    media_spread_30d = spread.rolling(720, min_periods=50).mean()
    df['spread_relativo'] = spread / media_spread_30d

    vol = df['Tick_Volume']
    media_30d = vol.rolling(720, min_periods=50).mean()
    std_30d = vol.rolling(720, min_periods=50).std(ddof=1)
    df['tick_volume_z'] = (vol - media_30d) / std_30d
    df['volumen_relativo'] = vol / media_30d

    return df


tablas = {valor: _variables_liquidez(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[1]} columnas — spread_relativo media {tabla['spread_relativo'].mean():.3f}, "
          f"volumen_relativo media {tabla['volumen_relativo'].mean():.3f}")

BTCUSD: 57 columnas — spread_relativo media 0.999, volumen_relativo media 1.005


## 6. Variables fundamentales de estado (Q1-10)

Las columnas `x2_score`/`x2_score_cross`/`x2_score_tendencia` ya están en la
tabla desde la Sección 1 (`merge_asof` backward = forward-fill del último
valor conocido). `x2_history.json` es muy escaso para BTCUSD (solo 2 registros
en todo el horizonte, ver `X5_alternativo.md` §2) — casi todo el historial
queda con el mismo score forward-filled. Las variables de esta sección
existen justamente para distinguir una actualización real de la mera
repetición del forward-fill:

- `fundamental_cambio` = `x2_score.diff()` — 0 en la inmensa mayoría de las
  filas (forward-fill), distinto de 0 solo donde llega una actualización real.
- `fundamental_direccion_cambio` = signo de `fundamental_cambio` (`+1`/`-1`/`0`).
- `fundamental_magnitud_cambio` = `|fundamental_cambio|`.
- `fundamental_edad` = velas H1 desde la última actualización real (mismo
  patrón de `groupby().cumcount()` que `duracion_drawdown` en la Sección 3);
  mide qué tan vieja es la información vigente.
- `fundamental_frecuencia_efectiva_90d` = fracción de velas, en una ventana
  rodante de 90 días (2160h — más ancha que los 30d del resto del notebook
  para tener chance real de capturar alguna actualización dado lo escaso de
  `x2_history`), que corresponden a una actualización real.
- `fundamental_estabilidad_90d` = desviación estándar rodante de `x2_score`
  en la misma ventana de 90 días — con forward-fill queda en 0 casi siempre y
  solo se mueve alrededor de una actualización; complementa a la frecuencia
  (cuenta eventos) con la magnitud del cambio.
- `fundamental_contradiccion` = `(x2_score_cross - 0.5) * (x2_score_tendencia - 0.5)`
  — ambos componentes están centrados en 0.5 (ver `X2_fundamentals.py`):
  negativo = contradicción (un componente por sobre neutro y el otro por
  debajo), positivo = consistencia, 0 = al menos uno es neutro.

In [8]:
def _variables_fundamentales(df):
    score = df['x2_score']
    cambio = score.diff()
    df['fundamental_cambio'] = cambio
    df['fundamental_direccion_cambio'] = np.sign(cambio)
    df['fundamental_magnitud_cambio'] = cambio.abs()

    actualizo = cambio.fillna(0) != 0
    grupo = actualizo.cumsum()
    df['fundamental_edad'] = df.groupby(grupo).cumcount()

    df['fundamental_frecuencia_efectiva_90d'] = actualizo.rolling(2160, min_periods=50).mean()
    df['fundamental_estabilidad_90d'] = score.rolling(2160, min_periods=50).std(ddof=1)

    df['fundamental_contradiccion'] = (df['x2_score_cross'] - 0.5) * (df['x2_score_tendencia'] - 0.5)

    return df


tablas = {valor: _variables_fundamentales(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    n_actualizaciones = int((tabla['fundamental_cambio'].fillna(0) != 0).sum())
    print(f"{valor}: {tabla.shape[1]} columnas — {n_actualizaciones} actualizaciones reales de x2_score, "
          f"fundamental_edad max {tabla['fundamental_edad'].max()} períodos, "
          f"fundamental_contradiccion media {tabla['fundamental_contradiccion'].mean():.4f}")

BTCUSD: 64 columnas — 0 actualizaciones reales de x2_score, fundamental_edad max 40282 períodos, fundamental_contradiccion media nan


## 7. Variables de ruido (Q1-6)

Todas sobre retornos logarítmicos horarios y en ventana de 30 días (720h,
igual que el resto de métricas de 30d del notebook), reutilizando
`x3_atr_pct` de X3 en vez de recalcular ATR:

- `ruido_cambios_signo_30d` = fracción de velas, en la ventana, donde el
  signo del retorno log cambia respecto a la vela anterior — proxy de
  zigzag/whipsaw frente a movimiento direccional sostenido.
- `ruido_ratio_movimiento_30d` = `suma(|retorno log|)` en la ventana /
  `|retorno log acumulado|` en la misma ventana — inverso del Efficiency
  Ratio de Kaufman: cercano a 1 = recorrido ≈ desplazamiento neto (movimiento
  eficiente), alto = mucho recorrido para poco desplazamiento neto (ruido).
- `ruido_mecha_30d` = promedio rodante de `(High-Low-|Close-Open|)/(High-Low)`
  por vela — mismo concepto que la fuerza del rechazo de X0
  (`f = 1-|Close-Open|/(High-Low)`, ver `CLAUDE.md` Paso 2) pero agregado en
  ventana como variable de estado en vez de por vela; alto = velas con mecha
  larga en relación a su rango, indecisión reciente.
- `ruido_atr_sin_direccion_30d` = `x3_atr_pct` promedio en la ventana /
  `|retorno log acumulado|` en la misma ventana — ATR alto con retorno
  acumulado bajo; complementa a `ruido_ratio_movimiento_30d` midiendo lo mismo
  con el rango típico suavizado (ATR) en vez de la suma vela a vela de
  retornos.

In [9]:
def _variables_ruido(df):
    close = df['Close']
    log_ret = np.log(close / close.shift(1))

    signo = np.sign(log_ret)
    df['ruido_cambios_signo_30d'] = (signo != signo.shift(1)).rolling(720, min_periods=50).mean()

    retorno_30d = np.log(close / close.shift(720))
    movimiento_total_30d = log_ret.abs().rolling(720, min_periods=50).sum()
    df['ruido_ratio_movimiento_30d'] = movimiento_total_30d / retorno_30d.abs().replace(0, np.nan)

    rango = df['High'] - df['Low']
    mecha = (rango - (df['Close'] - df['Open']).abs()) / rango.replace(0, np.nan)
    df['ruido_mecha_30d'] = mecha.rolling(720, min_periods=50).mean()

    df['ruido_atr_sin_direccion_30d'] = df['x3_atr_pct'].rolling(720, min_periods=50).mean() / retorno_30d.abs().replace(0, np.nan)

    return df


tablas = {valor: _variables_ruido(tabla) for valor, tabla in tablas.items()}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[1]} columnas — ruido_cambios_signo_30d media {tabla['ruido_cambios_signo_30d'].mean():.3f}, "
          f"ruido_mecha_30d media {tabla['ruido_mecha_30d'].mean():.3f}")

BTCUSD: 68 columnas — ruido_cambios_signo_30d media 0.529, ruido_mecha_30d media 0.572
